In [1]:
import torch

from torch_openreml import MixedModelREML
from torch_openreml.covariance import DummyMatrix, ScalarMatrix

n, p = 10, 2

group = ["a", "b"] * 5
effect = torch.tensor([1.0, -1.0]).repeat(n // 2)

X = torch.randn(n, p)
y = X @ torch.tensor([0.5, -0.5]) + effect + 0.5 * torch.randn(n)

reml = MixedModelREML(
    genotype=(DummyMatrix(group), ScalarMatrix(group)),
)

names, free_params = reml.optimize(y, X, verbose=1)
names, free_params

[  1] ∥∇∥:       5.5271, ∥Δ∥: 4.7602, η: 0.25, ∥Δᶜ∥: 1.1900, accepted: True , log 𝓛:  -0.0922
[  2] ∥∇∥:       0.7148, ∥Δ∥: 0.2877, η: 1.00, ∥Δᶜ∥: 0.2877, accepted: True , log 𝓛:  -0.0121
[  3] ∥∇∥:       0.2420, ∥Δ∥: 0.0566, η: 1.00, ∥Δᶜ∥: 0.0566, accepted: True , log 𝓛:  -0.0050
[  4] ∥∇∥:       0.0119, ∥Δ∥: 0.0031, η: 1.00, ∥Δᶜ∥: 0.0031, accepted: True , log 𝓛:  -0.0050
[  5] ∥∇∥:       0.0000, ∥Δ∥: 0.0000, η: 1.00, ∥Δᶜ∥: 0.0000, accepted: True , log 𝓛:  -0.0050

[∇: score, Δ: 𝐉⁻¹∇, η: step length, Δᶜ: clip(𝛉 + ηΔ, lb, ub) - 𝛉, 𝓛: restricted likelihood]

✓ Converged at iteration 5, total time: 0.01s



(['G/sigma^2', 'R/sigma^2'], tensor([-0.0048, -1.1646]))

In [2]:
history = reml.history
history["loglik"]

[tensor(-0.0922),
 tensor(-0.0121),
 tensor(-0.0050),
 tensor(-0.0050),
 tensor(-0.0050)]

In [3]:
torch.stack(history["loglik"])

tensor([-0.0922, -0.0121, -0.0050, -0.0050, -0.0050])

In [4]:
torch.stack(history["free_params"])

tensor([[ 0.2233, -1.1689],
        [-0.0645, -1.1644],
        [-0.0079, -1.1646],
        [-0.0048, -1.1646],
        [-0.0048, -1.1646]])

In [5]:
names, theta_hat = reml.get_free_params("last")
names, theta_hat

(['G/sigma^2', 'R/sigma^2'], tensor([-0.0048, -1.1646]))

In [6]:
params = reml.build_params("last", out_format="dict")
params

{'G/sigma^2': tensor([0.9905]), 'R/sigma^2': tensor([0.0974])}

In [7]:
tree = reml.tree("last")
tree

{'/': tensor([[1.0879, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905,
          0.0000],
         [0.0000, 1.0879, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000,
          0.9905],
         [0.9905, 0.0000, 1.0879, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905,
          0.0000],
         [0.0000, 0.9905, 0.0000, 1.0879, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000,
          0.9905],
         [0.9905, 0.0000, 0.9905, 0.0000, 1.0879, 0.0000, 0.9905, 0.0000, 0.9905,
          0.0000],
         [0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 1.0879, 0.0000, 0.9905, 0.0000,
          0.9905],
         [0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 1.0879, 0.0000, 0.9905,
          0.0000],
         [0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 1.0879, 0.0000,
          0.9905],
         [0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 1.0879,
          0.0000],
         [0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000, 0.9905, 0.0000,
    

In [8]:
beta_hat = reml.blue(y, X, theta_hat)
beta_hat

tensor([ 0.2577, -0.3913])

In [9]:
cov_beta = reml.cov_blue(X, theta_hat)
cov_beta

tensor([[ 0.0295, -0.0071],
        [-0.0071,  0.0223]])

In [10]:
se_beta = torch.sqrt(reml.var_blue(X, theta_hat))
se_beta

tensor([0.1718, 0.1494])

In [11]:
b_hat = reml.blup()
b_hat

tensor([ 0.9683, -0.9979])

In [12]:
b_cov = reml.cov_blup()
b_cov

tensor([[ 0.9682, -0.0035],
        [-0.0035,  0.9652]])

In [13]:
b_var = reml.var_blup()
b_var

tensor([0.9682, 0.9652])

In [14]:
b_by_term = reml.blup_per_term()
b_by_term

{'genotype': tensor([ 0.9683, -0.9979])}

In [15]:
b_cov_by_term = reml.cov_blup_per_term()
b_cov_by_term

{'genotype': tensor([[ 0.9682, -0.0035],
         [-0.0035,  0.9652]])}

In [16]:
b_var_by_term = reml.var_blup_per_term()
b_var_by_term

{'genotype': tensor([0.9682, 0.9652])}

In [17]:
y_marginal = reml.marginal_predict()
y_marginal

tensor([ 0.1906, -0.3714, -0.3499, -0.4857, -0.2634, -0.2160, -0.5087,  0.2695,
         0.1386, -0.0688])

In [18]:
y_conditional = reml.predict()
y_conditional

tensor([ 1.1589, -1.3693,  0.6184, -1.4836,  0.7048, -1.2139,  0.4595, -0.7284,
         1.1069, -1.0667])

In [19]:
y_new = reml.predict_new(
    X[:2],
    {"genotype": torch.tensor([[1.0, 0.0], [0.0, 1.0]])},
)
y_new

tensor([ 1.1589, -1.3693])

In [20]:
z_new = DummyMatrix(group).transform(["a", "b"])

y_new = reml.predict_new(
    X[:2],
    z_new,
)
y_new

tensor([ 1.1589, -1.3693])

In [21]:
e_marginal = reml.marginal_residual(y, X, theta_hat)
e_marginal

tensor([ 1.3035, -0.8965,  1.3673, -0.9385,  0.7651, -1.3004,  0.7445, -1.1696,
         0.7563, -0.7827])

In [22]:
e_conditional = reml.residual(y, X, theta_hat)
e_conditional

tensor([ 0.3352,  0.1014,  0.3990,  0.0594, -0.2032, -0.3024, -0.2238, -0.1717,
        -0.2120,  0.2152])

In [23]:
c11, c12, c21, c22 = reml.c_inverse_blocks(X, theta_hat)
c11, c12, c21, c22

(tensor([[ 0.0295, -0.0071],
         [-0.0071,  0.0223]]),
 tensor([[ 0.0051,  0.0127],
         [-0.0082, -0.0068]]),
 tensor([[ 0.0051, -0.0082],
         [ 0.0127, -0.0068]]),
 tensor([[0.0223, 0.0035],
         [0.0035, 0.0253]]))

In [24]:
loglik_value, loglik_method = reml.loglik(y, X, theta_hat)
loglik_value, loglik_method

(tensor(-0.0050), 'cform')

In [25]:
reml.is_converged()

True